# SQL Practice Lab

Comprehensive SQL practice using SQLite with pandas for visualization.

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
np.random.seed(42)

## 1. Create Database and Sample Tables

In [ ]:
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

# Create tables
cursor.executescript('''
CREATE TABLE users (
    user_id INTEGER PRIMARY KEY,
    name TEXT,
    email TEXT,
    signup_date DATE,
    country TEXT,
    plan TEXT
);

CREATE TABLE products (
    product_id INTEGER PRIMARY KEY,
    name TEXT,
    category TEXT,
    price REAL
);

CREATE TABLE orders (
    order_id INTEGER PRIMARY KEY,
    user_id INTEGER,
    product_id INTEGER,
    order_date DATE,
    quantity INTEGER,
    total_amount REAL,
    FOREIGN KEY (user_id) REFERENCES users(user_id),
    FOREIGN KEY (product_id) REFERENCES products(product_id)
);

CREATE TABLE events (
    event_id INTEGER PRIMARY KEY,
    user_id INTEGER,
    event_type TEXT,
    event_timestamp DATETIME,
    page TEXT,
    FOREIGN KEY (user_id) REFERENCES users(user_id)
);
''')

# Populate users
countries = ['US', 'UK', 'DE', 'FR', 'JP', 'IN', 'BR', 'CA']
plans = ['free', 'basic', 'premium']
users_data = []
base_date = datetime(2023, 1, 1)
for i in range(1, 201):
    signup = base_date + timedelta(days=np.random.randint(0, 365))
    users_data.append((i, f'User_{i}', f'user{i}@email.com', 
                       signup.strftime('%Y-%m-%d'),
                       np.random.choice(countries),
                       np.random.choice(plans, p=[0.5, 0.3, 0.2])))
cursor.executemany('INSERT INTO users VALUES (?,?,?,?,?,?)', users_data)

# Populate products
categories = ['Electronics', 'Books', 'Clothing', 'Food', 'Software']
products_data = []
for i in range(1, 51):
    cat = np.random.choice(categories)
    price = round(np.random.uniform(5, 500), 2)
    products_data.append((i, f'Product_{i}', cat, price))
cursor.executemany('INSERT INTO products VALUES (?,?,?,?)', products_data)

# Populate orders
orders_data = []
for i in range(1, 1001):
    uid = np.random.randint(1, 201)
    pid = np.random.randint(1, 51)
    order_date = base_date + timedelta(days=np.random.randint(0, 365))
    qty = np.random.randint(1, 5)
    price = products_data[pid-1][3]
    orders_data.append((i, uid, pid, order_date.strftime('%Y-%m-%d'), qty, round(price * qty, 2)))
cursor.executemany('INSERT INTO orders VALUES (?,?,?,?,?,?)', orders_data)

# Populate events (for sessionization)
event_types = ['page_view', 'click', 'scroll', 'purchase', 'signup']
pages = ['home', 'product', 'cart', 'checkout', 'profile', 'search']
events_data = []
eid = 1
for uid in range(1, 51):  # 50 users with events
    n_events = np.random.randint(10, 50)
    ts = base_date + timedelta(days=np.random.randint(0, 30))
    for _ in range(n_events):
        # Sometimes big gap (new session), sometimes small gap
        gap = np.random.choice([1, 2, 5, 60, 300, 3600], p=[0.3, 0.2, 0.2, 0.15, 0.1, 0.05])
        ts = ts + timedelta(minutes=gap)
        events_data.append((eid, uid, np.random.choice(event_types, p=[0.4, 0.25, 0.15, 0.1, 0.1]),
                           ts.strftime('%Y-%m-%d %H:%M:%S'), np.random.choice(pages)))
        eid += 1
cursor.executemany('INSERT INTO events VALUES (?,?,?,?,?)', events_data)
conn.commit()

print("Database created!")
for table in ['users', 'products', 'orders', 'events']:
    count = cursor.execute(f'SELECT COUNT(*) FROM {table}').fetchone()[0]
    print(f"  {table}: {count} rows")


## 2. Basic Queries: SELECT, WHERE, ORDER BY

In [ ]:
# Basic SELECT with filtering
query = '''
SELECT user_id, name, country, plan, signup_date
FROM users
WHERE plan = 'premium' AND country IN ('US', 'UK')
ORDER BY signup_date DESC
LIMIT 10;
'''
df = pd.read_sql_query(query, conn)
print("Premium users from US/UK (most recent first):")
print(df.to_string(index=False))


## 3. JOINs with Visualization

In [ ]:
# INNER JOIN: Users with their orders
query_inner = '''
SELECT u.name, u.country, COUNT(o.order_id) as order_count, 
       ROUND(SUM(o.total_amount), 2) as total_spent
FROM users u
INNER JOIN orders o ON u.user_id = o.user_id
GROUP BY u.user_id
ORDER BY total_spent DESC
LIMIT 10;
'''
print("TOP 10 CUSTOMERS (INNER JOIN - only users with orders):")
print(pd.read_sql_query(query_inner, conn).to_string(index=False))

# LEFT JOIN: All users, including those without orders
query_left = '''
SELECT u.plan, 
       COUNT(DISTINCT u.user_id) as total_users,
       COUNT(DISTINCT CASE WHEN o.order_id IS NOT NULL THEN u.user_id END) as users_with_orders,
       ROUND(100.0 * COUNT(DISTINCT CASE WHEN o.order_id IS NOT NULL THEN u.user_id END) / 
             COUNT(DISTINCT u.user_id), 1) as conversion_pct
FROM users u
LEFT JOIN orders o ON u.user_id = o.user_id
GROUP BY u.plan;
'''
print("\nCONVERSION BY PLAN (LEFT JOIN - includes users without orders):")
df_join = pd.read_sql_query(query_left, conn)
print(df_join.to_string(index=False))

# Visualize
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(df_join['plan'], df_join['conversion_pct'], color=['gray', 'steelblue', 'gold'])
ax.set_ylabel('Conversion %')
ax.set_title('Order Conversion Rate by Plan')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()


## 4. Aggregations and GROUP BY

In [ ]:
query = '''
SELECT p.category,
       COUNT(o.order_id) as num_orders,
       ROUND(SUM(o.total_amount), 2) as revenue,
       ROUND(AVG(o.total_amount), 2) as avg_order_value,
       COUNT(DISTINCT o.user_id) as unique_buyers
FROM orders o
JOIN products p ON o.product_id = p.product_id
GROUP BY p.category
HAVING num_orders > 50
ORDER BY revenue DESC;
'''
df_agg = pd.read_sql_query(query, conn)
print("Revenue by Category (HAVING > 50 orders):")
print(df_agg.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].barh(df_agg['category'], df_agg['revenue'], color='steelblue')
axes[0].set_xlabel('Revenue ($)')
axes[0].set_title('Revenue by Category')
axes[1].barh(df_agg['category'], df_agg['avg_order_value'], color='coral')
axes[1].set_xlabel('Avg Order Value ($)')
axes[1].set_title('Avg Order Value by Category')
plt.tight_layout()
plt.show()


## 5. Window Functions

In [ ]:
# ROW_NUMBER, RANK, running totals
query = '''
SELECT 
    user_id,
    order_date,
    total_amount,
    ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY order_date) as order_num,
    RANK() OVER (PARTITION BY user_id ORDER BY total_amount DESC) as amount_rank,
    SUM(total_amount) OVER (PARTITION BY user_id ORDER BY order_date 
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) as running_total,
    LAG(total_amount, 1) OVER (PARTITION BY user_id ORDER BY order_date) as prev_order_amount
FROM orders
WHERE user_id IN (1, 2, 3)
ORDER BY user_id, order_date
LIMIT 20;
'''
df_window = pd.read_sql_query(query, conn)
print("Window Functions Demo (ROW_NUMBER, RANK, Running Total, LAG):")
print(df_window.to_string(index=False))


In [ ]:
# Percentile rank of customers
query = '''
WITH customer_totals AS (
    SELECT user_id, SUM(total_amount) as lifetime_value
    FROM orders
    GROUP BY user_id
)
SELECT user_id, 
       ROUND(lifetime_value, 2) as ltv,
       NTILE(4) OVER (ORDER BY lifetime_value) as quartile,
       ROUND(PERCENT_RANK() OVER (ORDER BY lifetime_value) * 100, 1) as percentile
FROM customer_totals
ORDER BY lifetime_value DESC
LIMIT 15;
'''
print("Customer Lifetime Value with Percentile Rankings:")
print(pd.read_sql_query(query, conn).to_string(index=False))


## 6. CTEs (Common Table Expressions)

In [ ]:
# Multi-level CTE: Find users whose spending is above their country's average
query = '''
WITH user_spending AS (
    SELECT u.user_id, u.name, u.country, u.plan,
           COALESCE(SUM(o.total_amount), 0) as total_spent
    FROM users u
    LEFT JOIN orders o ON u.user_id = o.user_id
    GROUP BY u.user_id
),
country_avg AS (
    SELECT country, ROUND(AVG(total_spent), 2) as avg_spent
    FROM user_spending
    GROUP BY country
)
SELECT us.name, us.country, us.plan,
       ROUND(us.total_spent, 2) as spent,
       ca.avg_spent as country_avg,
       ROUND(us.total_spent - ca.avg_spent, 2) as vs_avg
FROM user_spending us
JOIN country_avg ca ON us.country = ca.country
WHERE us.total_spent > ca.avg_spent * 1.5
ORDER BY vs_avg DESC
LIMIT 10;
'''
print("Users spending 1.5x+ their country average (using CTEs):")
print(pd.read_sql_query(query, conn).to_string(index=False))


## 7. Subqueries (Correlated and Uncorrelated)

In [ ]:
# Uncorrelated subquery: Products never ordered
query_uncorr = '''
SELECT product_id, name, category, price
FROM products
WHERE product_id NOT IN (SELECT DISTINCT product_id FROM orders)
ORDER BY price DESC;
'''
print("Products never ordered (uncorrelated subquery):")
df = pd.read_sql_query(query_uncorr, conn)
print(df.to_string(index=False) if len(df) > 0 else "  All products have been ordered!")

# Correlated subquery: Each user's most expensive order
query_corr = '''
SELECT o.user_id, o.order_date, o.total_amount
FROM orders o
WHERE o.total_amount = (
    SELECT MAX(o2.total_amount)
    FROM orders o2
    WHERE o2.user_id = o.user_id
)
ORDER BY o.total_amount DESC
LIMIT 10;
'''
print("\nEach user's largest order (correlated subquery):")
print(pd.read_sql_query(query_corr, conn).to_string(index=False))


## 8. Sessionization Query

In [ ]:
# Group events into sessions (30-minute inactivity = new session)
query = '''
WITH event_gaps AS (
    SELECT *,
        LAG(event_timestamp) OVER (PARTITION BY user_id ORDER BY event_timestamp) as prev_ts,
        CAST((julianday(event_timestamp) - julianday(
            LAG(event_timestamp) OVER (PARTITION BY user_id ORDER BY event_timestamp)
        )) * 24 * 60 AS INTEGER) as gap_minutes
    FROM events
),
session_starts AS (
    SELECT *,
        CASE WHEN gap_minutes > 30 OR gap_minutes IS NULL THEN 1 ELSE 0 END as is_new_session
    FROM event_gaps
),
sessions AS (
    SELECT *,
        SUM(is_new_session) OVER (PARTITION BY user_id ORDER BY event_timestamp
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) as session_id
    FROM session_starts
)
SELECT user_id, session_id,
       MIN(event_timestamp) as session_start,
       MAX(event_timestamp) as session_end,
       COUNT(*) as events_in_session,
       COUNT(DISTINCT page) as pages_visited,
       COUNT(DISTINCT event_type) as event_types
FROM sessions
WHERE user_id <= 5
GROUP BY user_id, session_id
ORDER BY user_id, session_start
LIMIT 15;
'''
print("Session Analysis (30-min inactivity threshold):")
print(pd.read_sql_query(query, conn).to_string(index=False))


## 9. Cohort Analysis

In [ ]:
# Monthly cohort retention analysis
query = '''
WITH user_cohort AS (
    SELECT user_id, 
           strftime('%Y-%m', signup_date) as cohort_month
    FROM users
),
user_orders AS (
    SELECT o.user_id,
           uc.cohort_month,
           strftime('%Y-%m', o.order_date) as order_month,
           CAST((julianday(o.order_date) - julianday(uc.cohort_month || '-01')) / 30 AS INTEGER) as months_since_signup
    FROM orders o
    JOIN user_cohort uc ON o.user_id = uc.user_id
)
SELECT cohort_month,
       months_since_signup,
       COUNT(DISTINCT user_id) as active_users
FROM user_orders
WHERE months_since_signup BETWEEN 0 AND 6
GROUP BY cohort_month, months_since_signup
ORDER BY cohort_month, months_since_signup;
'''
df_cohort = pd.read_sql_query(query, conn)
print("Cohort Retention (first 6 months):")
print(df_cohort.head(20).to_string(index=False))

# Pivot for heatmap
pivot = df_cohort.pivot(index='cohort_month', columns='months_since_signup', values='active_users').fillna(0)
fig, ax = plt.subplots(figsize=(10, 6))
im = ax.imshow(pivot.values, aspect='auto', cmap='YlGnBu')
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index, fontsize=8)
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels([f'M+{c}' for c in pivot.columns])
ax.set_xlabel('Months Since Signup')
ax.set_ylabel('Cohort')
ax.set_title('Cohort Retention Heatmap')
plt.colorbar(im, label='Active Users')
plt.tight_layout()
plt.show()


## 10. Funnel Analysis

In [ ]:
# Conversion funnel: page_view -> click -> purchase
query = '''
WITH funnel AS (
    SELECT 
        user_id,
        MAX(CASE WHEN event_type = 'page_view' THEN 1 ELSE 0 END) as viewed,
        MAX(CASE WHEN event_type = 'click' THEN 1 ELSE 0 END) as clicked,
        MAX(CASE WHEN event_type = 'signup' THEN 1 ELSE 0 END) as signed_up,
        MAX(CASE WHEN event_type = 'purchase' THEN 1 ELSE 0 END) as purchased
    FROM events
    GROUP BY user_id
)
SELECT 
    SUM(viewed) as step1_view,
    SUM(clicked) as step2_click,
    SUM(signed_up) as step3_signup,
    SUM(purchased) as step4_purchase,
    ROUND(100.0 * SUM(clicked) / SUM(viewed), 1) as view_to_click_pct,
    ROUND(100.0 * SUM(signed_up) / SUM(clicked), 1) as click_to_signup_pct,
    ROUND(100.0 * SUM(purchased) / SUM(signed_up), 1) as signup_to_purchase_pct
FROM funnel;
'''
df_funnel = pd.read_sql_query(query, conn)
print("Conversion Funnel:")
print(df_funnel.T.to_string())

# Visualize funnel
steps = ['Page View', 'Click', 'Signup', 'Purchase']
values = [df_funnel['step1_view'].iloc[0], df_funnel['step2_click'].iloc[0],
          df_funnel['step3_signup'].iloc[0], df_funnel['step4_purchase'].iloc[0]]

fig, ax = plt.subplots(figsize=(8, 5))
colors = plt.cm.Blues(np.linspace(0.3, 0.9, len(steps)))
bars = ax.barh(steps[::-1], values[::-1], color=colors[::-1])
for bar, val in zip(bars, values[::-1]):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2, 
            f'{val} users', va='center')
ax.set_xlabel('Users')
ax.set_title('Conversion Funnel')
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()


## 11. Query Optimization Tips

In [ ]:
# EXPLAIN QUERY PLAN in SQLite
queries_to_explain = [
    ("Without index", "SELECT * FROM orders WHERE user_id = 5"),
    ("With subquery", "SELECT * FROM users WHERE user_id IN (SELECT user_id FROM orders WHERE total_amount > 500)"),
    ("Join query", "SELECT u.name, o.total_amount FROM users u JOIN orders o ON u.user_id = o.user_id WHERE o.total_amount > 100"),
]

print("EXPLAIN QUERY PLAN Analysis")
print("=" * 60)
for label, query in queries_to_explain:
    plan = cursor.execute(f"EXPLAIN QUERY PLAN {query}").fetchall()
    print(f"\n{label}:")
    print(f"  SQL: {query}")
    print(f"  Plan:")
    for row in plan:
        print(f"    {row}")

# Create an index and show improvement
print("\n" + "=" * 60)
print("After creating index on orders(user_id):")
cursor.execute("CREATE INDEX idx_orders_user ON orders(user_id)")
plan = cursor.execute("EXPLAIN QUERY PLAN SELECT * FROM orders WHERE user_id = 5").fetchall()
for row in plan:
    print(f"  {row}")

print("\nOptimization tips:")
print("  1. Index columns used in WHERE, JOIN, ORDER BY")
print("  2. Avoid SELECT * - specify needed columns")
print("  3. Use EXISTS instead of IN for correlated subqueries")
print("  4. Avoid functions on indexed columns in WHERE")
print("  5. Use LIMIT for exploration queries")


## 12. Feature Engineering with SQL

In [ ]:
# Time-windowed aggregations for ML features
query = '''
WITH user_features AS (
    SELECT 
        u.user_id,
        u.plan,
        u.country,
        -- Lifetime metrics
        COUNT(o.order_id) as total_orders,
        COALESCE(ROUND(SUM(o.total_amount), 2), 0) as lifetime_revenue,
        COALESCE(ROUND(AVG(o.total_amount), 2), 0) as avg_order_value,
        -- Recency
        CAST(julianday('2023-12-31') - julianday(MAX(o.order_date)) AS INTEGER) as days_since_last_order,
        -- Frequency
        CASE WHEN COUNT(o.order_id) > 0 THEN
            ROUND(CAST(julianday(MAX(o.order_date)) - julianday(MIN(o.order_date)) AS REAL) / 
                  NULLIF(COUNT(o.order_id) - 1, 0), 1)
        ELSE NULL END as avg_days_between_orders,
        -- Product diversity
        COUNT(DISTINCT o.product_id) as unique_products,
        -- Recent activity (last 90 days)
        SUM(CASE WHEN julianday('2023-12-31') - julianday(o.order_date) <= 90 
            THEN 1 ELSE 0 END) as orders_last_90d,
        COALESCE(SUM(CASE WHEN julianday('2023-12-31') - julianday(o.order_date) <= 90 
            THEN o.total_amount ELSE 0 END), 0) as revenue_last_90d
    FROM users u
    LEFT JOIN orders o ON u.user_id = o.user_id
    GROUP BY u.user_id
)
SELECT *
FROM user_features
WHERE total_orders > 0
ORDER BY lifetime_revenue DESC
LIMIT 15;
'''
df_features = pd.read_sql_query(query, conn)
print("ML Feature Engineering with SQL (time-windowed aggregations):")
print(df_features.to_string(index=False))


In [ ]:
# RFM Segmentation
query = '''
WITH rfm AS (
    SELECT user_id,
        CAST(julianday('2023-12-31') - julianday(MAX(order_date)) AS INTEGER) as recency,
        COUNT(order_id) as frequency,
        ROUND(SUM(total_amount), 2) as monetary
    FROM orders
    GROUP BY user_id
),
rfm_scores AS (
    SELECT *,
        NTILE(5) OVER (ORDER BY recency DESC) as r_score,
        NTILE(5) OVER (ORDER BY frequency) as f_score,
        NTILE(5) OVER (ORDER BY monetary) as m_score
    FROM rfm
)
SELECT *,
    CAST(r_score AS TEXT) || CAST(f_score AS TEXT) || CAST(m_score AS TEXT) as rfm_segment,
    CASE 
        WHEN r_score >= 4 AND f_score >= 4 THEN 'Champions'
        WHEN r_score >= 3 AND f_score >= 3 THEN 'Loyal'
        WHEN r_score >= 4 AND f_score <= 2 THEN 'New'
        WHEN r_score <= 2 AND f_score >= 3 THEN 'At Risk'
        ELSE 'Others'
    END as segment_name
FROM rfm_scores
ORDER BY monetary DESC
LIMIT 15;
'''
print("RFM Customer Segmentation:")
print(pd.read_sql_query(query, conn).to_string(index=False))


## Summary

SQL concepts covered:
- **Basic queries**: SELECT, WHERE, ORDER BY, LIMIT
- **JOINs**: INNER, LEFT (with visual explanation)
- **Aggregations**: COUNT, SUM, AVG, GROUP BY, HAVING
- **Window functions**: ROW_NUMBER, RANK, NTILE, LAG, running totals
- **CTEs**: Multi-level WITH clauses for readable complex queries
- **Subqueries**: Correlated and uncorrelated
- **Sessionization**: Gap-based session detection
- **Cohort analysis**: Retention over time
- **Funnel analysis**: Multi-step conversion tracking
- **Query optimization**: EXPLAIN, indexing
- **Feature engineering**: Time-windowed aggregations, RFM

In [ ]:
conn.close()
print("Database connection closed. Practice complete!")
